# COMP9085 Week 2 Laboratory
## Diagnostic Model Auditor: decision trees on real biological data

**Two hours. Bring your answers to the questions in bold; they are the point of
the session, not the code.**

You are auditing a model. By the end you will have grown a decision tree on
real diagnostic data, found where it stops generalising, looked inside it,
traced a real case through it, deployed it, and written the document that says
what it must not be used for.

**Dataset.** Wisconsin Diagnostic Breast Cancer (WDBC): 569 cases, 30 features,
212 malignant and 357 benign. Real benchmark data, used here for educational
machine-learning experimentation.

**This is not a clinical diagnostic system.** Nothing you build today is
validated for any clinical use, and Task 10 asks you to write that down
properly.

---

### Before you start

```
pip install numpy pandas scikit-learn matplotlib
```

Everything runs on a laptop in under a minute. No GPU, no internet after the
first import.

### Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

SEED = 42          # fix it once, here, and never silently change it
np.random.seed(SEED)

data = load_breast_cancer()
X = data.data
y = 1 - data.target          # recode so 1 = malignant, the clinical positive
feature_names = list(data.feature_names)
print("X", X.shape, "  y", y.shape)

## Task 1 — Meet the data

Before fitting anything, look at what you have. A model you cannot describe is
a model you cannot audit.

Report the shape, the class distribution, the feature types, and the
**majority-class baseline**: the accuracy you get by predicting the commoner
class for every case.

**Question 1.1.** What does one row of this matrix physically correspond to,
and who produced the numbers?

**Question 1.2.** Why must every accuracy you report later be compared against
the baseline rather than against 50 per cent?

In [ ]:
# Report: n_samples, n_features, counts of each class, and the baseline.
# Then look at the first few rows as a DataFrame.

# your code here


## Task 2 — Grow a tree

Split the data **once**, stratified, 70/30, with `random_state=SEED`. Fit a
`DecisionTreeClassifier` with `criterion="entropy"` and **no depth limit**.

Report the depth, the number of nodes and leaves, and the training and
held-out accuracy. Then **visualise the tree**.

**Question 2.1.** The training accuracy will be 100 per cent. Is that good
news?

**Question 2.2.** `DecisionTreeClassifier` is not an implementation of ID3.
Name two concrete differences you can see in the tree you just drew.

In [ ]:
# Split, fit an unrestricted entropy tree, report the numbers, plot the tree.

# your code here


## Task 3 — Investigate overfitting

Train a tree at every depth from 1 to 12. Record training and held-out
accuracy at each. Plot both curves on the same axes.

**Question 3.1.** At which depth do the curves separate, and what is happening
to the tree at that point?

**Question 3.2.** Which depth would you choose, and what is wrong with the way
you just chose it? (Task 8 fixes this.)

In [ ]:
# Sweep max_depth from 1 to 12 and plot the two accuracy curves.

# your code here


## Task 4 — Audit the actual tree

Fit a depth-3 tree and print it with `export_text`. Then answer, from the tree
itself rather than from intuition:

**Question 4.1.** What is the root feature and what threshold was selected?
What does that feature mean biologically?

**Question 4.2.** Which leaves are pure, and which are mixed? How many training
cases are in the smallest leaf?

**Question 4.3.** Is any branch so specific that you would not trust it on a
new patient? Say which, and why.

In [ ]:
# Fit max_depth=3, print it with export_text, plot it, and inspect the leaves.

# your code here


## Task 5 — Trace a real case

Take one case from the held-out set. **Before you run anything**, work out by
hand which leaf it will reach: read the root test, look up that feature's value
in the case, and follow the branch. Write your answer down.

Then use `decision_path` to check.

**Question 5.1.** Which tests did the tree apply, and in what order?

**Question 5.2.** How many of the 30 features were consulted? What happened to
the others?

In [ ]:
# Pick a held-out case, predict by hand, then verify with decision_path.

case_index = 6
case = X_test[case_index]

# your code here


## Task 6 — Deploy the model

Open the Week 2 page of the course website and scroll to **Deployment
demonstration**. It runs the same kind of tree, fitted in your browser on the
same data.

1. Press **Load example case** and read the prediction and the decision path.
2. Change one measurement by a small amount and analyse again.
3. Now change it until the prediction flips. How far did you have to go?
4. Enter something impossible, such as 900 for a perimeter, and see what the
   interface does.

**Question 6.1.** Which single measurement moved the prediction, and why that
one?

**Question 6.2.** The interface warns you when a value is outside the range
observed in the teaching dataset, but still returns a prediction. Is that the
right behaviour? Argue either way.

In [ ]:
# No code for this task. Write your answers in the cell below.


## Task 7 — Mystery case

Below is a case with its label hidden. **Predict it yourself first**, using the
depth-3 tree you printed in Task 4. Write your prediction down before running
the cell that reveals the answer.

**Question 7.1.** Which split changed your mind, if any?

**Question 7.2.** If the model and you disagree, which of you is more likely to
be wrong, and how would you find out?

In [ ]:
mystery = X_test[17]
print("some of the measurements:")
for f in ["worst perimeter", "worst concave points", "mean texture"]:
    print(f"  {f:24s} {mystery[feature_names.index(f)]:.4f}")

# Write your prediction here before running the next cell:
my_prediction = "benign"   # or "malignant"


## Task 8 — Validation, done properly

Task 3 chose a depth by looking at the held-out set. Do it honestly instead.

Split **60 / 20 / 20**: train, validation, final test. Choose `max_depth` on the
validation set. Then evaluate on the final test set **once**, and report the
confusion matrix.

**Question 8.1.** How does the final test accuracy compare with the validation
accuracy at the chosen depth? Which is the honest estimate?

**Question 8.2.** Why is repeatedly checking the final test set a problem, given
that you never train on it?

In [ ]:
# 60/20/20, choose the depth on validation, touch the final test set once.

# your code here


## Task 9 — Pruning

Compare an unpruned tree with a pruned one. Use
`cost_complexity_pruning_path` to get the candidate `ccp_alpha` values, fit a
tree at each, and plot accuracy and the number of leaves against alpha.

**Visualise both trees.**

**Question 9.1.** What does pruning cost you, and what does it buy?

**Question 9.2.** Choose alpha on the validation set from Task 8, not on the
test set. Does the honest choice agree with the one that maximises test
accuracy?

In [ ]:
# Cost-complexity pruning: path, sweep, plot, and draw both trees.

# your code here


## Task 10 — Model card

Write a model card for the depth-3 tree, in the markdown cell below. This is
the deliverable. It follows Mitchell et al. (2019).

Cover, in your own words:

- **Intended use.** What is this model for, and who would use it?
- **Data.** What is in it, where did it come from, when, and who is missing?
- **Features.** What are they, and how are they obtained?
- **Model.** Type, criterion, hyperparameters, how they were chosen, and the
  random seed.
- **Evidence.** Which numbers, measured on what, with what protocol.
- **Limitations.** Where does it fail, and how would you know?
- **Risks.** What are the consequences of each error type, and who bears them?
- **Why this is not a clinical system.** Be specific.

**Be honest about the protocol you actually followed**, including anywhere you
looked at the test set. A model card that describes an idealised process is
worse than none.

### Model card — decision tree for WDBC diagnosis

*(replace this cell with your model card)*

**Intended use**

**Data**

**Features**

**Model**

**Evidence**

**Limitations**

**Risks**

**Why this is not a clinical system**


---

## Before you leave

Check that you can answer these without looking anything up.

1. Why did the tree ask about the root feature first? Give the number.
2. What is the difference between the decision path and an explanation?
3. Which accuracy from this notebook would you put in a report, and which would you refuse to?
4. Name one thing that would have to be true before this model could be used on a patient, and say whether it is true.
